
# 04 — Cálculo das Idades Críticas — UTKFace

Este notebook tem como objetivo analisar o desempenho das tecnologias DeepFace e MiVOLO nas proximidades dos pontos de corte etários utilizados no estudo, considerando os resultados obtidos a partir da base de imagens UTKFace.

A análise considera janelas de idade de referência próximas aos limiares de 12, 16, 17 e 18 anos, permitindo observar o comportamento do erro de estimativa e a disponibilidade de resultados numéricos nessas regiões.

São realizadas as seguintes atividades:

- Seleção dos registros de acordo com as janelas de idade de referência;
- Cálculo do Erro Absoluto Médio (MAE) em cada janela;
- Contabilização das saídas numéricas disponíveis;
- Identificação das estimativas negativas;
- Cálculo da cobertura de estimativas numéricas e de decisões válidas;
- Comparação dos resultados entre DeepFace e MiVOLO.

As janelas consideradas são:

- Limiar de 12 anos: idades de referência de 10 a 14 anos;
- Limiar de 16 anos: idades de referência de 14 a 18 anos;
- Limiar de 17 anos: idades de referência de 15 a 19 anos;
- Limiar de 18 anos: idades de referência de 16 a 20 anos.

O limiar de 17 anos é tratado como análise complementar.

O MAE considera todas as estimativas numéricas finitas, inclusive valores negativos. Entretanto, estimativas negativas não são consideradas decisões válidas para o cálculo da cobertura de decisões.

Este notebook concentra-se na análise do erro e da disponibilidade das estimativas nas janelas próximas aos pontos de corte. As métricas de classificação e as matrizes de confusão são apresentadas separadamente no Notebook 05.


In [ ]:

from google.colab import drive

drive.mount("/content/drive")


In [ ]:

## 1. Bibliotecas, caminhos e parâmetros

Nesta etapa são importadas as bibliotecas necessárias e definidos os diretórios dos arquivos CSV gerados pelos notebooks de inferência do DeepFace e do MiVOLO.


In [ ]:

import os
import numpy as np
import pandas as pd

# ============================================================
# DIRETÓRIOS DOS RESULTADOS
# ============================================================

PASTA_DEEPFACE = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados/deepface"
)

PASTA_MIVOLO = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/resultados/MiVOLO"
)

PASTA_SAIDA = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados/analises_idades_criticas"
)

os.makedirs(PASTA_SAIDA, exist_ok=True)

PARTES = ["part1", "part2", "part3"]

ARQUIVOS = {
    "DeepFace": {
        parte: os.path.join(
            PASTA_DEEPFACE,
            f"resultados_deepface_utkface_{parte}.csv"
        )
        for parte in PARTES
    },
    "MiVOLO": {
        parte: os.path.join(
            PASTA_MIVOLO,
            f"resultados_mivolo_utkface_{parte}.csv"
        )
        for parte in PARTES
    }
}

STATUS_SUCESSO = {
    "DeepFace": "Sucesso",
    "MiVOLO": "OK"
}

COLUNAS_OBRIGATORIAS = [
    "Arquivo",
    "Idade Real",
    "Idade Estimada",
    "Erro Absoluto",
    "Status"
]

print("Configuração concluída.")
print("Diretório de saída:", PASTA_SAIDA)


In [ ]:

## 2. Carregamento e preparação dos resultados

São carregados os arquivos CSV das três partições da UTKFace para cada tecnologia.

Os registros são preparados para a análise, considerando o status de sucesso da inferência e a validade numérica das idades. O erro absoluto é recalculado a partir das idades de referência e estimada, preservando estimativas negativas quando forem numéricas e finitas.


In [ ]:

dados = []

for tecnologia, arquivos_parte in ARQUIVOS.items():

    for parte, caminho in arquivos_parte.items():

        if not os.path.isfile(caminho):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {caminho}"
            )

        df = pd.read_csv(caminho)

        faltantes = [
            coluna for coluna in COLUNAS_OBRIGATORIAS
            if coluna not in df.columns
        ]

        if faltantes:
            raise ValueError(
                f"Colunas ausentes em {caminho}: {faltantes}"
            )

        df = df[COLUNAS_OBRIGATORIAS].copy()

        df["Tecnologia"] = tecnologia
        df["Parte"] = parte

        for coluna in [
            "Idade Real",
            "Idade Estimada",
            "Erro Absoluto"
        ]:
            df[coluna] = pd.to_numeric(
                df[coluna],
                errors="coerce"
            )

        df["Status_Sucesso"] = (
            df["Status"].astype(str).str.strip()
            == STATUS_SUCESSO[tecnologia]
        )

        df["Idade_Referencia_Valida"] = (
            np.isfinite(df["Idade Real"])
        )

        df["Saida_Numerica"] = (
            df["Status_Sucesso"]
            & df["Idade_Referencia_Valida"]
            & np.isfinite(df["Idade Estimada"])
        )

        df["Decisao_Valida"] = (
            df["Saida_Numerica"]
            & (df["Idade Estimada"] >= 0)
        )

        df["Erro_Absoluto_Recalculado"] = np.where(
            df["Saida_Numerica"],
            (
                df["Idade Estimada"]
                - df["Idade Real"]
            ).abs(),
            np.nan
        )

        dados.append(df)

df_resultados = pd.concat(
    dados,
    ignore_index=True
)

print("Carregamento concluído.")
print("Total de registros:", len(df_resultados))

display(
    df_resultados.head()
)


In [ ]:

## 3. Definição das janelas próximas aos pontos de corte

A análise considera intervalos de dois anos abaixo e dois anos acima de cada ponto de corte, incluindo os limites inferior e superior.

As janelas são selecionadas com base na idade de referência registrada no nome das imagens da UTKFace. O limiar de 17 anos é incluído como análise complementar.


In [ ]:

JANELAS_CRITICAS = {
    12: {
        "Idade_Minima": 10,
        "Idade_Maxima": 14,
        "Tipo": "Principal"
    },
    16: {
        "Idade_Minima": 14,
        "Idade_Maxima": 18,
        "Tipo": "Principal"
    },
    17: {
        "Idade_Minima": 15,
        "Idade_Maxima": 19,
        "Tipo": "Complementar"
    },
    18: {
        "Idade_Minima": 16,
        "Idade_Maxima": 20,
        "Tipo": "Principal"
    }
}

print("Janelas configuradas:")

for limiar, configuracao in JANELAS_CRITICAS.items():

    print(
        f"Limiar {limiar} anos: "
        f"{configuracao['Idade_Minima']} a "
        f"{configuracao['Idade_Maxima']} anos "
        f"({configuracao['Tipo']})"
    )


In [ ]:

## 4. Cálculo do MAE e da cobertura nas janelas críticas

Para cada tecnologia, partição e janela etária, são calculados:

- Quantidade total de registros com idade de referência válida;
- Quantidade de saídas numéricas;
- Quantidade de estimativas negativas;
- Quantidade de decisões válidas;
- Quantidade de registros sem decisão válida;
- Erro Absoluto Médio (MAE);
- Cobertura de saídas numéricas;
- Cobertura de decisões válidas.

O MAE é calculado somente sobre as saídas numéricas finitas, incluindo estimativas negativas. Já a cobertura de decisões válidas considera apenas estimativas numéricas não negativas.


In [ ]:

resultados_janelas = []

for tecnologia in ["DeepFace", "MiVOLO"]:

    df_tecnologia = df_resultados[
        df_resultados["Tecnologia"] == tecnologia
    ].copy()

    for limiar, configuracao in JANELAS_CRITICAS.items():

        idade_minima = configuracao["Idade_Minima"]
        idade_maxima = configuracao["Idade_Maxima"]

        # Seleção pela idade de referência.
        df_janela = df_tecnologia[
            df_tecnologia["Idade_Referencia_Valida"]
            & (
                df_tecnologia["Idade Real"]
                >= idade_minima
            )
            & (
                df_tecnologia["Idade Real"]
                <= idade_maxima
            )
        ].copy()

        grupos = [
            (
                parte,
                df_janela[
                    df_janela["Parte"] == parte
                ].copy()
            )
            for parte in PARTES
        ]

        grupos.append(
            ("Consolidado", df_janela.copy())
        )

        for parte, grupo in grupos:

            total = len(grupo)

            saidas_numericas = grupo[
                grupo["Saida_Numerica"]
            ]

            decisoes_validas = grupo[
                grupo["Decisao_Valida"]
            ]

            estimativas_negativas = int(
                (
                    saidas_numericas["Idade Estimada"] < 0
                ).sum()
            )

            mae = (
                saidas_numericas[
                    "Erro_Absoluto_Recalculado"
                ].mean()
                if len(saidas_numericas) > 0
                else np.nan
            )

            resultados_janelas.append({

                "Tecnologia": tecnologia,

                "Limiar": limiar,

                "Tipo_Limiar": configuracao["Tipo"],

                "Idade_Minima": idade_minima,

                "Idade_Maxima": idade_maxima,

                "Parte": parte,

                "Total_Registros": total,

                "Saidas_Numericas": len(saidas_numericas),

                "Estimativas_Negativas": estimativas_negativas,

                "Decisoes_Validas": len(decisoes_validas),

                "Sem_Decisao_Valida": (
                    total - len(decisoes_validas)
                ),

                "MAE_Anos": mae,

                "Cobertura_Numerica_Pct": (
                    100 * len(saidas_numericas) / total
                    if total > 0 else np.nan
                ),

                "Cobertura_Decisoes_Validas_Pct": (
                    100 * len(decisoes_validas) / total
                    if total > 0 else np.nan
                )
            })

df_idades_criticas = pd.DataFrame(
    resultados_janelas
)

print("Cálculo das janelas concluído.")

display(
    df_idades_criticas.head(12)
)


In [ ]:

## 5. Organização e apresentação dos resultados

Os resultados são organizados por tecnologia, limiar e partição. A linha denominada "Consolidado" representa o cálculo realizado diretamente sobre todos os registros das três partições reunidas, evitando a utilização de médias simples entre partições.


In [ ]:

ordem_tecnologias = ["DeepFace", "MiVOLO"]

ordem_partes = [
    "part1",
    "part2",
    "part3",
    "Consolidado"
]

df_idades_criticas["Tecnologia"] = pd.Categorical(
    df_idades_criticas["Tecnologia"],
    categories=ordem_tecnologias,
    ordered=True
)

df_idades_criticas["Parte"] = pd.Categorical(
    df_idades_criticas["Parte"],
    categories=ordem_partes,
    ordered=True
)

df_idades_criticas = (
    df_idades_criticas
    .sort_values(
        ["Tecnologia", "Limiar", "Parte"]
    )
    .reset_index(drop=True)
)

# Tabela formatada para visualização.
df_visualizacao = df_idades_criticas.copy()

colunas_arredondar = [
    "MAE_Anos",
    "Cobertura_Numerica_Pct",
    "Cobertura_Decisoes_Validas_Pct"
]

df_visualizacao[colunas_arredondar] = (
    df_visualizacao[colunas_arredondar].round(2)
)

display(df_visualizacao)


In [ ]:

## 6. Comparação consolidada entre as tecnologias

Nesta etapa são apresentados exclusivamente os resultados consolidados das três partições, permitindo comparar o MAE e a cobertura das tecnologias em cada janela etária.


In [ ]:

df_comparacao = df_visualizacao[
    df_visualizacao["Parte"] == "Consolidado"
].copy()

df_comparacao = df_comparacao[
    [
        "Tecnologia",
        "Limiar",
        "Tipo_Limiar",
        "Idade_Minima",
        "Idade_Maxima",
        "Total_Registros",
        "Saidas_Numericas",
        "Estimativas_Negativas",
        "Decisoes_Validas",
        "Sem_Decisao_Valida",
        "MAE_Anos",
        "Cobertura_Numerica_Pct",
        "Cobertura_Decisoes_Validas_Pct"
    ]
].reset_index(drop=True)

display(df_comparacao)


In [ ]:

## 7. Exportação dos resultados

As tabelas calculadas são exportadas em formato CSV para armazenamento e utilização nas análises e na elaboração dos resultados do trabalho.


In [ ]:

arquivo_completo = os.path.join(
    PASTA_SAIDA,
    "resultados_idades_criticas_utkface.csv"
)

arquivo_consolidado = os.path.join(
    PASTA_SAIDA,
    "resultados_idades_criticas_consolidados_utkface.csv"
)

df_idades_criticas.to_csv(
    arquivo_completo,
    index=False,
    encoding="utf-8-sig"
)

df_comparacao.to_csv(
    arquivo_consolidado,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos exportados com sucesso:")
print(arquivo_completo)
print(arquivo_consolidado)


In [ ]:

## 8. Critérios para interpretação dos resultados

- As janelas são definidas pela idade de referência, com limites inclusivos.
- O limiar de 17 anos é complementar; os limiares de 12, 16 e 18 anos são os principais.
- O MAE considera todas as saídas numéricas finitas de inferências bem-sucedidas, inclusive estimativas negativas.
- Estimativas negativas não são consideradas decisões válidas.
- A cobertura numérica corresponde à proporção de registros da janela que possuem saída numérica finita.
- A cobertura de decisões válidas corresponde à proporção de registros da janela com estimativa numérica não negativa.
- Os registros sem decisão válida permanecem no denominador das coberturas.
- O MAE consolidado é calculado diretamente sobre todas as saídas numéricas das três partições reunidas.
- As janelas não representam tolerância de erro nem critério de acerto da estimativa.
- Este notebook não calcula matrizes de confusão ou métricas de classificação por limiar, pois essas análises são realizadas no Notebook 05.
